In [1]:
from ipynb.fs.defs.q_network import QNetwork

import torch
import torch.nn as nn
import torch.optim as optim

from collections import deque
import random

In [2]:
class ReplayBuffer():

    def __init__(self,capacity):
        self.memory = deque(maxlen=capacity)

    def push(self,transition):
        self.memory.append(transition)

    def sample(self,batch_size):
        return random.sample(self.memory,batch_size)

    def __len__(self):
        return len(self.memory)

class DQNAgent:

    def __init__(
        self,
        gamma=0.99,
        batch_size=32,
        epsilon=1.0,
        epsilon_decay=0.99,
        epsilon_min=0.01,
        buffer_size=10000,
        learning_rate=0.001
    ):

        self.policy_net = QNetwork()
        self.target_net = QNetwork()

        self.target_net.load_state_dict(
            self.policy_net.state_dict()
        )

        self.optimizer = optim.Adam(
            self.policy_net.parameters(),
            lr=learning_rate
        )

        self.replay_buffer = ReplayBuffer(
            buffer_size
        )

        self.gamma = gamma
        self.batch_size = batch_size

        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def choose_action(self, state):

        position_fraction = state[5]

        valid_actions = [0, 1, 2, 3, 4]

        if position_fraction <= 0:

            valid_actions.remove(3)
            valid_actions.remove(4)

        elif position_fraction >= 0.99:

            valid_actions.remove(1)
            valid_actions.remove(2)

        if random.random() < self.epsilon:

            return random.choice(valid_actions)

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        with torch.no_grad():

            q_values = (
                self.policy_net(state_tensor)
                .squeeze()
            )

        best_action = valid_actions[0]
        best_q = q_values[best_action]

        for action in valid_actions:

            if q_values[action] > best_q:

                best_q = q_values[action]
                best_action = action

        return best_action

    def train_step(self):

        if len(self.replay_buffer) < self.batch_size:
            return

        transitions = self.replay_buffer.sample(self.batch_size)
        states, actions, rewards, next_states, dones = zip(*transitions)

        states = torch.tensor(states,dtype=torch.float32)

        actions = torch.tensor(
            actions,dtype=torch.long).unsqueeze(1)
        rewards = torch.tensor(rewards,dtype=torch.float32)

        next_states = torch.tensor(
            next_states,
            dtype=torch.float32
        )

        dones = torch.tensor(
            dones,
            dtype=torch.float32
        )

        q_values = self.policy_net(states)

        current_q = q_values.gather(
            1,
            actions
        )

        with torch.no_grad():

            next_q_values = self.target_net(
                next_states
            )

            for i in range(len(next_states)):

                position_fraction = (next_states[i][5])

                if position_fraction <= 0:

                    next_q_values[i,3] = float("-inf")
                    next_q_values[i,4] = float("-inf")

                elif position_fraction >= 0.99:

                    next_q_values[i,1] = float("-inf")
                    next_q_values[i,2] = float("-inf")

            best_future_q = (
                next_q_values
                .max(dim=1)
                .values
            )

            target_q = (
                rewards
                + (1 - dones)
                * self.gamma
                * best_future_q
            )

        target_q = target_q.unsqueeze(1)

        loss = nn.MSELoss()(
            current_q,
            target_q
        )

        self.optimizer.zero_grad()

        loss.backward()

        self.optimizer.step()

    def update_target_net(self):

        self.target_net.load_state_dict(
            self.policy_net.state_dict()
        )
        
   


In [3]:
from ipynb.fs.defs.q_network import QNetwork

print(QNetwork)


<class 'ipynb.fs.defs.q_network.QNetwork'>
